In [11]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import os

PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = Path.cwd()

MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

model_path = MODEL_DIR / "vaac_tiny_best.keras"

QUANT_DIR = RESULTS_DIR / "quantized"
QUANT_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Model:", model_path)
print("Model exists:", model_path.exists())
print("Quantization directory:", QUANT_DIR)

TensorFlow: 2.21.0
Model: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\models\vaac_tiny_best.keras
Model exists: True
Quantization directory: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized


In [12]:
model = tf.keras.models.load_model(model_path)

print("FP32 model loaded successfully")
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)
print("Parameters  :", model.count_params())

FP32 model loaded successfully
Input shape : (None, 12000, 1)
Output shape: (None, 4)
Parameters  : 4244


In [13]:
WINDOW_DIR = PROJECT_ROOT / "data" / "processed" / "CWRU" / "windows"

TRAIN_METADATA = WINDOW_DIR / "train_metadata.csv"

train_df = pd.read_csv(TRAIN_METADATA)

print("Training metadata shape:", train_df.shape)
display(train_df.head())

Training metadata shape: (232, 8)


,recording_id,source_file,signal_id,class,window_id,start_sample,end_sample,label
0,IR007_0_X105,IR007_0.mat,X105,Inner Race,0,0,12000,2
1,IR007_0_X105,IR007_0.mat,X105,Inner Race,1,6000,18000,2
2,IR007_0_X105,IR007_0.mat,X105,Inner Race,2,12000,24000,2
3,IR007_0_X105,IR007_0.mat,X105,Inner Race,3,18000,30000,2
4,IR007_0_X105,IR007_0.mat,X105,Inner Race,4,24000,36000,2


In [14]:
TRAIN_WINDOW_DIR = WINDOW_DIR / "train"

print("Training window directory:")
print(TRAIN_WINDOW_DIR)

print("Exists:", TRAIN_WINDOW_DIR.exists())

files = list(TRAIN_WINDOW_DIR.glob("*.npy"))

print("Number of .npy files:", len(files))

print("\nExample files:")
for f in files[:5]:
    print(f.name)

Training window directory:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\train
Exists: True
Number of .npy files: 390

Example files:
B007_3_X121_window_0000.npy
B007_3_X121_window_0001.npy
B007_3_X121_window_0002.npy
B007_3_X121_window_0003.npy
B007_3_X121_window_0004.npy


In [15]:
def get_window_path(row):
    filename = (
        f"{row['recording_id']}"
        f"_window_{int(row['window_id']):04d}.npy"
    )
    
    return TRAIN_WINDOW_DIR / filename

In [16]:
example_path = get_window_path(train_df.iloc[0])

print(example_path)
print("Exists:", example_path.exists())

c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\train\IR007_0_X105_window_0000.npy
Exists: True


In [17]:
for i in range(min(10, len(train_df))):
    path = get_window_path(train_df.iloc[i])
    
    print(
        i,
        train_df.iloc[i]["recording_id"],
        train_df.iloc[i]["window_id"],
        "→",
        path.name,
        "Exists:",
        path.exists()
    )

0 IR007_0_X105 0 → IR007_0_X105_window_0000.npy Exists: True
1 IR007_0_X105 1 → IR007_0_X105_window_0001.npy Exists: True
2 IR007_0_X105 2 → IR007_0_X105_window_0002.npy Exists: True
3 IR007_0_X105 3 → IR007_0_X105_window_0003.npy Exists: True
4 IR007_0_X105 4 → IR007_0_X105_window_0004.npy Exists: True
5 IR007_0_X105 5 → IR007_0_X105_window_0005.npy Exists: True
6 IR007_0_X105 6 → IR007_0_X105_window_0006.npy Exists: True
7 IR007_0_X105 7 → IR007_0_X105_window_0007.npy Exists: True
8 IR007_0_X105 8 → IR007_0_X105_window_0008.npy Exists: True
9 IR007_0_X105 9 → IR007_0_X105_window_0009.npy Exists: True


In [18]:
CALIBRATION_SAMPLES = 100

calibration_df = train_df.sample(
    n=min(CALIBRATION_SAMPLES, len(train_df)),
    random_state=42
).reset_index(drop=True)

print("Calibration samples:", len(calibration_df))

Calibration samples: 100


In [19]:
def representative_dataset():
    for _, row in calibration_df.iterrows():
        
        path = get_window_path(row)
        
        signal = np.load(path).astype(np.float32)
        
        # Ensure shape = (1, 12000, 1)
        signal = signal.reshape(1, 12000, 1)
        
        yield [signal]

In [20]:
sample = next(representative_dataset())[0]

print("Representative sample shape:", sample.shape)
print("Datatype:", sample.dtype)
print("Minimum:", sample.min())
print("Maximum:", sample.max())

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\armaa\\OneDrive\\Desktop\\TinyML-Vibration-Anomaly-Detection\\data\\processed\\CWRU\\windows\\train\\OR007@6_2_X132_window_0006.npy'

In [21]:
# Step 224.5A — Find actual training window directories

CWRU_PROCESSED = PROJECT_ROOT / "data" / "processed" / "CWRU"

print("CWRU processed directory:")
print(CWRU_PROCESSED)

print("\nCandidate directories:")

for d in CWRU_PROCESSED.iterdir():
    if d.is_dir():
        print("-", d)

CWRU processed directory:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU

Candidate directories:
- c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\baseline_features
- c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\baseline_features_corrected
- c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\figures
- c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows
- c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected


In [22]:
# Step 224.5B — Find all training .npy files

all_npy_files = list(
    CWRU_PROCESSED.rglob("*.npy")
)

print("Total .npy files found:", len(all_npy_files))

print("\nFirst 20 files:")

for f in all_npy_files[:20]:
    print(f)

Total .npy files found: 1166

First 20 files:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0000.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0001.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0002.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0003.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0004.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0005.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0006.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-D

In [23]:
# Step 224.6A — Build an index of all available window files

window_file_index = {}

for path in all_npy_files:
    window_file_index[path.name] = path

print("Indexed window files:", len(window_file_index))

print("\nExample indexed files:")

for name, path in list(window_file_index.items())[:10]:
    print(name)
    print(" ->", path)

Indexed window files: 583

Example indexed files:
B007_0_X118_window_0000.npy
 -> c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected\train\B007_0_X118_window_0000.npy
B007_0_X118_window_0001.npy
 -> c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected\train\B007_0_X118_window_0001.npy
B007_0_X118_window_0002.npy
 -> c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected\train\B007_0_X118_window_0002.npy
B007_0_X118_window_0003.npy
 -> c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected\train\B007_0_X118_window_0003.npy
B007_0_X118_window_0004.npy
 -> c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected\train\B007_0_X118_window_0004.npy
B007_0_X118_window_0005.npy
 -> c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detect

In [24]:
# Step 224.6B — Robust window path lookup

def get_window_path(row):
    
    filename = (
        f"{row['recording_id']}"
        f"_window_{int(row['window_id']):04d}.npy"
    )
    
    if filename not in window_file_index:
        raise FileNotFoundError(
            f"\nWindow file not found:\n"
            f"{filename}\n\n"
            f"Recording ID: {row['recording_id']}\n"
            f"Window ID: {row['window_id']}"
        )
    
    return window_file_index[filename]

In [25]:
# Step 224.6C — Test problematic window

test_row = calibration_df[
    calibration_df["recording_id"] == "OR007@6_2_X132"
]

display(test_row.head())

,recording_id,source_file,signal_id,class,window_id,start_sample,end_sample,label
0,OR007@6_2_X132,OR007@6_2.mat,X132,Outer Race,6,36000,48000,3
10,OR007@6_2_X132,OR007@6_2.mat,X132,Outer Race,3,18000,30000,3
18,OR007@6_2_X132,OR007@6_2.mat,X132,Outer Race,14,84000,96000,3
48,OR007@6_2_X132,OR007@6_2.mat,X132,Outer Race,17,102000,114000,3
56,OR007@6_2_X132,OR007@6_2.mat,X132,Outer Race,2,12000,24000,3


In [26]:
for _, row in test_row.head(3).iterrows():
    
    path = get_window_path(row)
    
    print("Recording:", row["recording_id"])
    print("Window:", row["window_id"])
    print("File:", path)
    print("Exists:", path.exists())
    print()

Recording: OR007@6_2_X132
Window: 6
File: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected\train\OR007@6_2_X132_window_0006.npy
Exists: True

Recording: OR007@6_2_X132
Window: 3
File: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected\train\OR007@6_2_X132_window_0003.npy
Exists: True

Recording: OR007@6_2_X132
Window: 14
File: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected\train\OR007@6_2_X132_window_0014.npy
Exists: True



In [27]:
# Step 224.7 — Test representative dataset

sample = next(representative_dataset())[0]

print("Representative sample shape:", sample.shape)
print("Datatype:", sample.dtype)
print("Minimum:", sample.min())
print("Maximum:", sample.max())

Representative sample shape: (1, 12000, 1)
Datatype: float32
Minimum: -2.736626
Maximum: 2.8600767


In [28]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

print("Starting INT8 conversion...")

quantized_model = converter.convert()

print("INT8 conversion successful.")
print("Quantized model bytes:", len(quantized_model))

Starting INT8 conversion...
INFO:tensorflow:Assets written to: C:\Users\armaa\AppData\Local\Temp\tmpnrhuscrj\assets


INFO:tensorflow:Assets written to: C:\Users\armaa\AppData\Local\Temp\tmpnrhuscrj\assets


Saved artifact at 'C:\Users\armaa\AppData\Local\Temp\tmpnrhuscrj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 12000, 1), dtype=tf.float32, name='vibration_input')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1852192668688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1852213414720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1852213419648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1852213425632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1852213425808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1852213427040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1852216181152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1852216182384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1852216184320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1852216185376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  18522

c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\.venv\lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 conversion successful.
Quantized model bytes: 21848


In [29]:
INT8_MODEL_PATH = QUANT_DIR / "vaac_tiny_int8.tflite"

with open(INT8_MODEL_PATH, "wb") as f:
    f.write(quantized_model)

print("Saved INT8 model:")
print(INT8_MODEL_PATH)

Saved INT8 model:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized\vaac_tiny_int8.tflite


In [30]:
int8_file_bytes = os.path.getsize(INT8_MODEL_PATH)

int8_file_kb = int8_file_bytes / 1024
int8_file_mb = int8_file_kb / 1024

print("Actual INT8 TFLite Model Size")
print("=" * 45)

print(f"Bytes : {int8_file_bytes:,}")
print(f"KB    : {int8_file_kb:.2f}")
print(f"MB    : {int8_file_mb:.4f}")

Actual INT8 TFLite Model Size
Bytes : 21,848
KB    : 21.34
MB    : 0.0208


In [31]:
interpreter = tf.lite.Interpreter(
    model_path=str(INT8_MODEL_PATH)
)

interpreter.allocate_tensors()

print("INT8 TFLite interpreter initialized.")

INT8 TFLite interpreter initialized.


c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [32]:
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("INPUT")
print("=" * 40)

for detail in input_details:
    print("Name :", detail["name"])
    print("Shape:", detail["shape"])
    print("Type :", detail["dtype"])
    print("Quantization:", detail["quantization"])

print("\nOUTPUT")
print("=" * 40)

for detail in output_details:
    print("Name :", detail["name"])
    print("Shape:", detail["shape"])
    print("Type :", detail["dtype"])
    print("Quantization:", detail["quantization"])

INPUT
Name : serving_default_vibration_input:0
Shape: [    1 12000     1]
Type : <class 'numpy.int8'>
Quantization: (0.027604417875409126, -5)

OUTPUT
Name : StatefulPartitionedCall_1:0
Shape: [1 4]
Type : <class 'numpy.int8'>
Quantization: (0.00390625, -128)


In [33]:
input_scale, input_zero_point = (
    input_details[0]["quantization"]
)

output_scale, output_zero_point = (
    output_details[0]["quantization"]
)

print("Input quantization")
print("Scale      :", input_scale)
print("Zero point :", input_zero_point)

print("\nOutput quantization")
print("Scale      :", output_scale)
print("Zero point :", output_zero_point)

Input quantization
Scale      : 0.027604417875409126
Zero point : -5

Output quantization
Scale      : 0.00390625
Zero point : -128


In [34]:
all_tensors = interpreter.get_tensor_details()

float_tensors = []
int8_tensors = []
uint8_tensors = []

for tensor in all_tensors:
    dtype = tensor["dtype"]
    
    if dtype == np.float32:
        float_tensors.append(tensor["name"])
    elif dtype == np.int8:
        int8_tensors.append(tensor["name"])
    elif dtype == np.uint8:
        uint8_tensors.append(tensor["name"])

print("Tensor datatype summary")
print("=" * 45)

print("Float32 tensors:", len(float_tensors))
print("INT8 tensors   :", len(int8_tensors))
print("UINT8 tensors  :", len(uint8_tensors))

if float_tensors:
    print("\nFloat32 tensors:")
    for name in float_tensors:
        print("-", name)

Tensor datatype summary
Float32 tensors: 0
INT8 tensors   : 34
UINT8 tensors  : 0


In [35]:
fp32_model_bytes = os.path.getsize(model_path)

print("Model Size Comparison")
print("=" * 45)

print(
    f"Keras FP32 file : "
    f"{fp32_model_bytes / 1024:.2f} KB"
)

print(
    f"TFLite INT8 file: "
    f"{int8_file_bytes / 1024:.2f} KB"
)

Model Size Comparison
Keras FP32 file : 108.24 KB
TFLite INT8 file: 21.34 KB


In [36]:
file_reduction = (
    1 - int8_file_bytes / fp32_model_bytes
) * 100

print(
    f"\nFile-size difference: "
    f"{file_reduction:.2f}%"
)


File-size difference: 80.29%


In [37]:
quantization_info = pd.DataFrame({
    "Property": [
        "Original Parameters",
        "Input Shape",
        "Output Shape",
        "Input Dtype",
        "Output Dtype",
        "Input Scale",
        "Input Zero Point",
        "Output Scale",
        "Output Zero Point",
        "FP32 Keras File KB",
        "INT8 TFLite File KB"
    ],
    "Value": [
        model.count_params(),
        str(model.input_shape),
        str(model.output_shape),
        str(input_details[0]["dtype"]),
        str(output_details[0]["dtype"]),
        input_scale,
        input_zero_point,
        output_scale,
        output_zero_point,
        fp32_model_bytes / 1024,
        int8_file_bytes / 1024
    ]
})

quantization_info

,Property,Value
0,Original Parameters,4244
1,Input Shape,"(None, 12000, 1)"
2,Output Shape,"(None, 4)"
3,Input Dtype,<class 'numpy.int8'>
4,Output Dtype,<class 'numpy.int8'>
5,Input Scale,0.027604
6,Input Zero Point,-5
7,Output Scale,0.003906
8,Output Zero Point,-128
9,FP32 Keras File KB,108.240234


In [38]:
quantization_info_path = (
    QUANT_DIR / "vaac_tiny_int8_quantization_info.csv"
)

quantization_info.to_csv(
    quantization_info_path,
    index=False
)

print("Saved:")
print(quantization_info_path)

Saved:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized\vaac_tiny_int8_quantization_info.csv


In [39]:
print("=" * 60)
print("STEP 224 — INT8 QUANTIZATION")
print("=" * 60)

print("Original model parameters:", model.count_params())

print(
    "\nINT8 model:",
    INT8_MODEL_PATH
)

print(
    "INT8 file size:",
    f"{int8_file_kb:.2f} KB"
)

print(
    "\nInput dtype:",
    input_details[0]["dtype"]
)

print(
    "Output dtype:",
    output_details[0]["dtype"]
)

print(
    "\nInput scale:",
    input_scale
)

print(
    "Input zero point:",
    input_zero_point
)

print(
    "\nOutput scale:",
    output_scale
)

print(
    "Output zero point:",
    output_zero_point
)

print("\nStep 224 INT8 quantization completed.")

STEP 224 — INT8 QUANTIZATION
Original model parameters: 4244

INT8 model: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized\vaac_tiny_int8.tflite
INT8 file size: 21.34 KB

Input dtype: <class 'numpy.int8'>
Output dtype: <class 'numpy.int8'>

Input scale: 0.027604417875409126
Input zero point: -5

Output scale: 0.00390625
Output zero point: -128

Step 224 INT8 quantization completed.
